# Extracting interaction profiles

Interaction profiles (surface point cloud, ESP on a surface point cloud, and pharmacophores) can be generated for any arbitrary conformer.

In [ ]:
import open3d # open3d can occasionally cause issues during imports; importing it first can help alleviate that
from rdkit import Chem
from shepherd_score.conformer_generation import embed_conformer_from_smiles, charges_from_single_point_conformer_with_xtb
from shepherd_score.extract_profiles import get_atomic_vdw_radii, get_molecular_surface
from shepherd_score.extract_profiles import get_pharmacophores, get_electrostatic_potential, get_pharmacophores_dict

Get a conformer with RDKit embedding and MMFF94 optimization then obtain charges from xTB.

In [ ]:
ref_mol = embed_conformer_from_smiles('c1Cc2ccc(Cl)cc2C(=O)c1c3cc(N1nnc2cc(C)c(Cl)cc2c1=O)ccc3', MMFF_optimize=True)
partial_charges = charges_from_single_point_conformer_with_xtb(ref_mol)

## Extract surface
The vdW radii and the positions of each atom is used to generate a surface. Here we sample 200 points on the solvent accessible surface with a probe radius of 1.2 A. Note that the sampling is stochastic.

In [ ]:
radii = get_atomic_vdw_radii(ref_mol)
surface = get_molecular_surface(ref_mol.GetConformer().GetPositions(), radii, num_points=200, probe_radius=1.2)
surface.shape

## Extract electrostatic potential (ESP)
The ESP at each surface point is computed from the partial charges.

In [ ]:
esp = get_electrostatic_potential(ref_mol, partial_charges, surface)
esp.shape

## Extract pharmacophores

Pharmacophores are extracted using pre-defined SMARTS patterns given an RDKit Mol object. Using `get_pharmacophores` returns a `Pharmacophore` container that still unpacks as a tuple of NumPy arrays: `(pharm_types, pharm_positions, pharm_vectors)`. `get_pharmacophores_dict` is a helper function that provides an easy to interpret version. Non-directional pharmacophores (e.g., hydrophobe) have vectors set to the zero vector.

In [ ]:
get_pharmacophores_dict(ref_mol, multi_vector=False)

In [ ]:
pharm_types, pharm_pos, pharm_vecs = get_pharmacophores(ref_mol, multi_vector=False)

In [ ]:
pharm_types.shape, pharm_pos.shape, pharm_vecs.shape

The pharmacophore types are ordered (with 0 indexing) of the following list.

In [ ]:
from shepherd_score.score.constants import P_TYPES
P_TYPES

### Priority labels

`get_pharmacophores` returns a `Pharmacophore` container that still unpacks as `(types, positions, vectors)`. You can also mark each pharmacophore as priority (`1`) or not (`0`) from a set of atom indices. This is useful when only part of a binding site should drive downstream scoring (e.g. conditional evaluation with `priority_pharm_indices`).

**Labeling rules**

- Point features (Acceptor, Donor, Halogen, Cation, Anion, ZnBinder): label `1` if any contributing atom is in `priority_atoms`.
- Aromatic and ring-derived Hydrophobe: require at least `min_ring_priority_atoms` heavy ring atoms (default `3`) from a shared ring to appear in `priority_atoms`. Use `min_ring_priority_atoms=1` for looser labeling.

In [ ]:
import numpy as np
from shepherd_score.score.constants import P_TYPES

# Eager: compute labels in one call
priority_atoms = [a.GetIdx() for a in ref_mol.GetAtoms()
                  if a.GetAtomicNum() > 1 and a.IsInRing()][:3]
pharm = get_pharmacophores(ref_mol, multi_vector=False, priority_atoms=priority_atoms)
labels = pharm.labels

# Still unpacks as the original 3-tuple
pharm_types, pharm_pos, pharm_vecs = pharm

# Inspect labels by pharmacophore type
for idx, label in zip(pharm.types, labels):
    print(P_TYPES[int(idx)], label)

# Lazy alternative: retain atom ids, then label against multiple priority sets
pharm = get_pharmacophores(ref_mol, multi_vector=False, return_atom_ids=True)
labels_alt = pharm.priority_labels(priority_atoms)
np.array_equal(labels, labels_alt)

## Abstracted objects

We introduce the `Molecule` object that facilitates automatic generation of interaction profiles. It holds these profiles as attributes, alongside the RDKit `Mol` object. 

In [ ]:
from shepherd_score.container import Molecule

The same procedure is used as above to extract each interaction profile. Pass charges explicitly to select a charge model. `charge_model="mmff"` requests MMFF94; the default attempts xTB and warns before falling back to MMFF94 if xTB is unavailable.

In [ ]:
molec = Molecule(ref_mol, num_surf_points=200, probe_radius=1.2,
                 partial_charges=partial_charges, pharm_multi_vector=False)

In [ ]:
molec.surf_pos.shape, molec.surf_esp.shape

In [ ]:
molec.pharm_types.shape, molec.pharm_ancs.shape, molec.pharm_vecs.shape

In [ ]:
molec.mol # rdkit mol object